# DS 208 &middot; Programming for Data Science &mdash; Week 9 Lab
## Reading Files, SQL &amp; APIs

Data rarely arrives as a clean CSV. This week you load it from **JSON**, from a **SQL
database**, and from a live **web API** &mdash; and every path ends in the same DataFrame.

**How long:** about 45 minutes. The API part needs internet (Colab has it).

Work top to bottom. The Stretch section at the end is optional.

---
## Part 0 &middot; Read JSON into a DataFrame

JSON keeps structure that a flat CSV loses. `pd.read_json` turns a list of records
straight into a table. Run the cell.

In [ ]:
import pandas as pd
from io import StringIO

json_text = '[{"city":"Cebu","region":"Visayas","pop":964000},' \
            '{"city":"Davao","region":"Mindanao","pop":1777000},' \
            '{"city":"Baguio","region":"Luzon","pop":366000}]'

df = pd.read_json(StringIO(json_text))
print(df)
print("\ndtypes:\n", df.dtypes)

**Answer here** (double-click to edit):

1. Each JSON record became one row and each key became a column. Why is that mapping so
   convenient for tabular data?
   &rarr; *your answer*

2. Unlike a CSV, the JSON kept the numbers as numbers. Check `df.dtypes` &mdash; what type
   is `pop`, and why does that save you a conversion step?
   &rarr; *your answer*

---
## Part 1 &middot; Query a SQL database

A database scales where a CSV breaks. You ask it questions in **SQL** and load the
answer with `pd.read_sql`. We build a tiny SQLite database in memory. Run the cell.

In [ ]:
import sqlite3, pandas as pd

conn = sqlite3.connect(":memory:")
pd.DataFrame({
    "city": ["Cebu", "Davao", "Iloilo", "Manila"],
    "region": ["Visayas", "Mindanao", "Visayas", "Luzon"],
    "pop": [964000, 1777000, 457000, 1780000],
}).to_sql("cities", conn, index=False)

out = pd.read_sql("SELECT city, pop FROM cities WHERE region = 'Visayas'", conn)
print(out)

**Answer here:**

1. The `WHERE region = 'Visayas'` clause ran inside the database, not in pandas. Why is it
   often better to filter in SQL before the data reaches Python?
   &rarr; *your answer*

2. `read_sql` returned a familiar object. What type is `out`, and what does that mean for
   the pandas skills you already have?
   &rarr; *your answer*

---
## Part 2 &middot; Fetch from a web API

An **API** is a URL built for programs. You send a request, check the status, and parse
the JSON reply. Run the cell (needs internet).

In [ ]:
import requests, pandas as pd

r = requests.get("https://jsonplaceholder.typicode.com/users", timeout=30)
print("status:", r.status_code)          # 200 means OK

people = r.json()                          # parsed JSON -> list of dicts
df = pd.DataFrame(people)[["id", "name", "username", "email"]]
print(df.head())

**Answer here:**

1. Why check `r.status_code` *before* calling `r.json()`? What could go wrong if you
   skipped that check?
   &rarr; *your answer*

2. The reply was JSON, and `pd.DataFrame(...)` turned it into a table. How is this the same
   pattern as Part 0, just with the data coming over the network?
   &rarr; *your answer*

---
## Part 3 &middot; Every reader has a writer

Once data is a DataFrame you can save it in any format. Readers and writers mirror each
other. Run the cell.

In [ ]:
import pandas as pd
df = pd.DataFrame({"city": ["Cebu", "Davao"], "pop": [964000, 1777000]})

df.to_csv("cities_out.csv", index=False)
print(open("cities_out.csv").read())
print("round-trip equal:", pd.read_csv("cities_out.csv").equals(df))

**Answer here:**

1. `index=False` kept the row numbers out of the file. What would the CSV look like with
   `index=True`, and when might you actually want that?
   &rarr; *your answer*

2. The round-trip compared the reloaded frame to the original. Why is a save-then-reload
   check a good habit before you trust an export?
   &rarr; *your answer*

---
## Stretch &mdash; optional

Stop here if you like; the required part is done.

### Stretch 1 &middot; Filter in SQL

Write a `read_sql` query that returns only cities with `pop > 900000`, sorted from
largest to smallest (`ORDER BY pop DESC`).

In [ ]:
# your code here

### Stretch 2 &middot; One field from the API

From the API response in Part 2, build a DataFrame of just each user's `name` and the
`city` nested under `address` (hint: `person["address"]["city"]`).

In [ ]:
# your code here

---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to
download.

You need a **submit token** &mdash; one covers every lab for a month. Open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token), sign in and generate it,
then add it **once** to Colab's Secrets panel (the &#128273; icon, left sidebar) as
`LATARAK_TOKEN`. After that the cell reads it automatically, with no prompt. No Secrets
panel? The cell will just ask, hiding what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "ds208", 9

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/ds208/lab/9/submit"
    )

# The LIVE notebook, including edits you have not saved yet.
nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]

# A month-long token. Store it once in Colab Secrets (key LATARAK_TOKEN) and
# this reads it with no prompt; otherwise it asks and hides what you type.
try:
    from google.colab import userdata
    token = (userdata.get("LATARAK_TOKEN") or "").strip()
except Exception:
    token = ""
if not token:
    token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 9 submission page](https://portal.latarak.com/course/ds208/lab/9/submit) and upload it.

Re-submitting replaces your previous attempt; the most recent version is the one kept.